# Phase 9D PF-ERI Metric Learning Template

Do not claim success from training loss. Only held-out retrieval metrics count.


## 1. Mount Google Drive
Upload or mount the repo package. Do not upload delayed second-review files.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Install Requirements


In [ ]:
%pip install -r colab/phase9d_metric_learning/requirements_colab.txt


## 3. Set Project Path


In [ ]:
import os
PROJECT_PATH = "/content/drive/MyDrive/felid-review-readiness-triage"
os.chdir(PROJECT_PATH)
print(os.getcwd())


## 4. Load Manifest and Verify Inputs


In [ ]:
import pandas as pd
from pathlib import Path
manifest = pd.read_csv("outputs/czechlynx/phase9/pf_eri_metric_learning_prep/phase9d_image_training_manifest_internal.csv")
print(manifest.shape)
print(manifest[["split_id", "train_val_test_role"]].value_counts().sort_index())
print("missing image paths", (~manifest["image_path_internal"].map(lambda p: Path(p).exists())).sum())


## 5. Train Required Groups
Run each group. Training loss is diagnostic only.


In [ ]:
configs = [
    "config_baseline_all_images.yaml",
    "config_random_same_size.yaml",
    "config_quality_only.yaml",
    "config_pf_eri_selected.yaml",
    "config_pf_eri_weighted.yaml",
]
for cfg in configs:
    print("TRAIN", cfg)
    !python colab/phase9d_metric_learning/train_metric_learning.py --config colab/phase9d_metric_learning/{cfg}


## 6. Evaluate Held-Out Retrieval
Only these metrics count for scientific interpretation.


In [ ]:
import yaml
from pathlib import Path
for cfg in configs:
    cfg_path = Path("colab/phase9d_metric_learning") / cfg
    config = yaml.safe_load(cfg_path.read_text())
    ckpt = Path(config["output_dir"]) / "projection_head.pt"
    print("EVALUATE", cfg)
    !python colab/phase9d_metric_learning/evaluate_reid_retrieval.py --config {cfg_path} --checkpoint {ckpt}


## 7. Export Results


In [ ]:
!find outputs/czechlynx/phase9/pf_eri_metric_learning_results -type f | sort
!zip -r phase9d_colab_outputs.zip outputs/czechlynx/phase9/pf_eri_metric_learning_results


## Required Warning
Do not claim PF-ERI improves Re-ID learning unless held-out retrieval metrics beat all-images, random same-size, quality-only, and Phase 9B-R no-training reference baselines.
